# Chapter 13 — Unsupervised Learning

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 13. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
from itertools import combinations

TRANSACTIONS = [
    {"bread", "milk"}, {"bread", "butter", "milk"}, {"bread", "butter"},
    {"milk", "coffee"}, {"bread", "milk", "coffee"}, {"bread", "butter", "milk"},
    {"butter", "milk"}, {"bread", "butter", "milk", "coffee"},
]

def support(itemset):
    """Fraction of receipts that contain every item in the itemset."""
    return sum(itemset <= t for t in TRANSACTIONS) / len(TRANSACTIONS)

def apriori(min_support):
    """Frequent itemsets, level by level (Apriori property)."""
    items = sorted({a for t in TRANSACTIONS for a in t})
    level = [frozenset([a]) for a in items
             if support({a}) >= min_support]
    frequent = list(level)
    k = 2
    while level:
        candidates = {a | b for a in level for b in level
                      if len(a | b) == k}
        # pruning: every subset must be frequent
        candidates = [c for c in candidates
                      if all(frozenset(s) in frequent
                             for s in combinations(c, k - 1))]
        level = [c for c in candidates if support(c) >= min_support]
        frequent += level
        k += 1
    return frequent

def rules(frequent, min_confidence):
    result = []
    for itemset in frequent:
        for r in range(1, len(itemset)):
            for antecedent in combinations(sorted(itemset), r):
                a = frozenset(antecedent)
                c = itemset - a
                confidence = support(itemset) / support(a)
                if confidence >= min_confidence:
                    lift = confidence / support(c)
                    result.append((sorted(a), sorted(c), confidence, lift))
    return sorted(result, key=lambda x: (-round(x[3], 3), x[0], x[1]))

frequent = apriori(min_support=0.375)
print("frequent itemsets:", len(frequent))
for a, c, confidence, lift in rules(frequent, min_confidence=0.7):
    print(f"{', '.join(a)} → {', '.join(c)}: "
          f"confidence {confidence:.2f}, lift {lift:.2f}")

## Expected output

If everything works, you should see:

```
frequent itemsets: 9
coffee → milk: confidence 1.00, lift 1.14
butter → bread: confidence 0.80, lift 1.07
butter, milk → bread: confidence 0.75, lift 1.00
bread → milk: confidence 0.83, lift 0.95
milk → bread: confidence 0.71, lift 0.95
butter → milk: confidence 0.80, lift 0.91
bread, butter → milk: confidence 0.75, lift 0.86
```

Now try changing a number in the code and run it again to see what happens.